# Projet Leyenda - Livrable 3
##### *Par Noah.V, Noah.M, Hugo.L*

## **Contexte:**
L'entreprise TouNum est une entreprise qui travaille sur la numérisation de différents type de documents. Ils essayent de moderniser leur architecture pour le tri automatique de documents due au grand nombre de documents fourni par les clients. Faisant appel à nous pour determiner le captionning des images avec légendes de manière automatiques. Nous remarquons deux contraintes, la première est la qualité des images qui diffère par exemple entre chaque image, et trié automatiquement si c'est une photo ou d'autres types de documents.

Le workflow demandé par l'entreprise devra suivre ce modèle la:

![Image demandé](src/workflow.png)

Le projet sera divisé en trois partie:
- La première sera de contrsuire un CNN via des datasets d'images pré étiqueté
- Le deuxième sera d'améliorer la clarté et l'enlever les différents bruits ou flou des images
- Le troisème sera de créer un réseau de neurones qui génère des légendes pour des photographies

Après avoir réalisé les deux premiers livrables, l'objectif est de créer un réseau de neurones qui génère des légendes pour des photographies, en s'appuyant sur le dataset dataset MS COCO. Il sera divisé en 2 partie la premiere via un CNN et la deuxieme sera le RNN pour le captionning. Nous utilisons un modèle pré-entrainé: *Inception V3*

Détail du ***Livrable 3:***
- L'architecture schématique complète du réseau utilisé pour le captioning explicitant le type de CNN utilisé pour les prétraitements.
- Un petit descriptif sur le pré-traitements de images et du texte.
- Le code explicitant l'architecture du CNN et du RNN utilisés dans le captioning.
- L'évolution sous forme de courbes des performances du réseau pendant l'entrainement. Affichage de quelques exemples pour les tests.

## Importation des librairies

In [20]:

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, UpSampling2D
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.models import Model
import os
from pathlib import Path
import json
import collections
import random



## Etape 1 : Chargement des images


### Importation du dataset et annotation COCO

In [15]:
# Chemin du fichier d'annotations
annotation_folder = os.path.abspath('.')+"/data/annotations/"
annotation_file = annotation_folder+"captions_train2014.json"

# Chemin du dossier contenant les images à annoter
image_folder = '/data/train2014/'
PATH = os.path.abspath('.') + image_folder

print("annotation file ", annotation_file)
print("le chemin PATH ", PATH)

annotation file  c:\Users\hmjhl\Documents\GitHub\Cnn\Livrable 3/data/annotations/captions_train2014.json
le chemin PATH  c:\Users\hmjhl\Documents\GitHub\Cnn\Livrable 3/data/train2014/


In [16]:
with open(annotation_folder+"captions_val2014.json", 'r') as f:
    annotations = json.load(f)

with open(annotation_folder+"instances_val2014.json", 'r') as f:
    instances = json.load(f)

In [18]:
# Lecture du fichier d'annotation
with open(annotation_file, 'r') as f:
    annotations = json.load(f)

# Grouper toutes les annotations ayant le meme identifiant.
image_path_to_caption = collections.defaultdict(list)
for val in annotations['annotations']:
    # marquer le debut et la fin de chaque annotation
    caption =  val['caption']
    # L'identifiant d'une image fait partie de son chemin d'accès
    image_path = PATH + 'COCO_train2014_' + '%012d.jpg' % (val['image_id'])
    # Rajout du caption associé à image_path
    #A COMPLETER
    image_path_to_caption[image_path].append(caption)


In [21]:
image_paths = list(image_path_to_caption.keys())
random.shuffle(image_paths)
train_image_paths = image_paths[:2000]
print(len(train_image_paths))

2000


In [ ]:
# Liste de toutes les annotations
train_captions = []
# Liste de tous les noms de fichiers des images dupliquées (en nombre d'annotations par image)
img_name_vector = []

for image_path in train_image_paths:
    caption_list = image_path_to_caption[image_path]
    # Rajout de caption_list dans train_captions
    train_captions.append(caption_list)
    # Rajout de image_path dupliquée len(caption_list) fois
    [img_name_vector.append(image_path) for _ in range(len(caption_list))]



### Importation du Model InceptionV3

In [ ]:
model_inc = keras.applications.InceptionV3(include_top=False,
                                                weights='imagenet',
                                                input_shape=(225, 225, 3))#modifié taille img

model_inc.trainable = False



### Chargements des variables et Hyperparametres.

Ses différentes valeurs seront utilisés tout au long de ce notebook pour les models.

### CNN

